# DREAM AI — train our native model on Kaggle

This notebook trains DREAM's own PyTorch transformer from random initialization. It does **not** download or call another model, use a third-party AI API, or import pretrained weights.

Kaggle offers free notebook GPU accelerators when available; access can be queued or limited. Select a GPU in Notebook Settings, enable Internet so the notebook can clone DREAM's public repository, and attach a **private Kaggle Dataset** containing your reviewed `approved-records.jsonl`. See [Kaggle notebook setup](https://www.kaggle.com/docs/notebooks) and [GPU usage guidance](https://www.kaggle.com/docs/efficient-gpu-usage).

**Important:** this is an early research-scale model, not a capable general assistant. A real, high-quality, rights-reviewed corpus is required. Do not set rights/privacy/quality flags to true unless a person has actually completed those checks. GPU availability and notebook sessions are not guaranteed.


**Five-minute daily sessions:** each run budgets up to 4 minutes for native pretraining and 1 minute for SFT when reviewed SFT data is available. The trainer saves a resumable checkpoint at the time limit. Kaggle does not automatically carry `/kaggle/working` into a new session: the setup cell now checks the private DREAM R2 vault first, verifies the newest bundle's SHA-256, ZIP CRC and per-checkpoint hashes, and restores matching checkpoints automatically. Kaggle input ZIP restoration remains as a fallback. Keep this dataset private.

In [ ]:
sft_completed = False
from pathlib import Path
import os, subprocess, sys, torch, json, shutil, hashlib
from zipfile import ZipFile

REPO = Path('/kaggle/working/dream-ai')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/mjnr0477/dream-ai.git', str(REPO)], check=True)
os.chdir(REPO)

# Kaggle preserves notebook outputs when you save a version.
# Keep checkpoints and the final artifact bundle under /kaggle/working.
CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PRETRAIN_CHECKPOINT = CHECKPOINT_DIR / 'dream-bpe.pt'
SFT_CHECKPOINT = CHECKPOINT_DIR / 'dream-bpe-instruct.pt'
SFT_BASE_CHECKPOINT = CHECKPOINT_DIR / 'dream-bpe-sft-base.pt'

# DREAM R2 recovery: restore the newest verified native-training bundle automatically.
from kaggle_secrets import UserSecretsClient
import requests

DREAM_HOST = "https://dream-host-live.onrender.com"
R2_ARTIFACT_TOKEN = ""
try:
    R2_ARTIFACT_TOKEN = UserSecretsClient().get_secret("DREAM_NOTEBOOK_ARTIFACT_TOKEN").strip()
except Exception:
    pass
R2_HEADERS = {"Authorization": "Bearer " + R2_ARTIFACT_TOKEN} if R2_ARTIFACT_TOKEN else {}
r2_bundle_restored = False
r2_bundle_created_at = ""

def r2_upload_file(file_path, artifact_name):
    file_path = Path(file_path)
    if not R2_ARTIFACT_TOKEN:
        print("R2 backup skipped: DREAM_NOTEBOOK_ARTIFACT_TOKEN is not configured.")
        return None
    digest = hashlib.sha256()
    with file_path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    expected_sha = digest.hexdigest()
    with file_path.open("rb") as handle:
        response = requests.put(
            f"{DREAM_HOST}/api/v1/notebook/artifacts/{artifact_name}",
            data=handle,
            headers={**R2_HEADERS, "Content-Type": "application/octet-stream"},
            timeout=(20, 1800)
        )
    response.raise_for_status()
    metadata = response.json()["artifact"]
    if metadata.get("sha256") != expected_sha or metadata.get("size_bytes") != file_path.stat().st_size:
        raise RuntimeError("R2 upload metadata does not match the local checkpoint.")
    print("R2 checkpoint backup verified:", artifact_name, expected_sha)
    return metadata

if R2_ARTIFACT_TOKEN:
    listing = requests.get(
        DREAM_HOST + "/api/v1/notebook/artifacts",
        headers=R2_HEADERS, timeout=(20, 120)
    )
    listing.raise_for_status()
    bundles = [x for x in listing.json().get("artifacts", [])
               if x.get("name") == "dream-training-artifacts.zip"]
    if bundles:
        latest = max(bundles, key=lambda x: x.get("created_at", ""))
        r2_bundle_created_at = latest.get("created_at", "")
        recovered_zip = Path("/kaggle/working/dream-training-artifacts-r2.zip")
        response = requests.get(
            DREAM_HOST + "/api/v1/notebook/artifacts/" + latest["sha256"],
            headers=R2_HEADERS, stream=True, timeout=(20, 1800)
        )
        response.raise_for_status()
        digest = hashlib.sha256()
        with recovered_zip.open("wb") as handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    digest.update(chunk)
                    handle.write(chunk)
        if digest.hexdigest() != latest["sha256"]:
            raise RuntimeError("R2 recovery bundle SHA-256 mismatch.")
        with ZipFile(recovered_zip) as archive:
            if archive.testzip() is not None:
                raise RuntimeError("R2 recovery bundle failed ZIP CRC validation.")
            sums = {}
            for line in archive.read("SHA256SUMS.txt").decode("utf-8").splitlines():
                expected, separator, name = line.partition("  ")
                if separator and len(expected) == 64:
                    sums[name] = expected
            for name, destination in (
                ("checkpoints/dream-bpe.pt", PRETRAIN_CHECKPOINT),
                ("checkpoints/dream-bpe-instruct.pt", SFT_CHECKPOINT),
                ("checkpoints/dream-bpe-sft-base.pt", SFT_BASE_CHECKPOINT)
            ):
                if name not in archive.namelist():
                    continue
                payload = archive.read(name)
                if hashlib.sha256(payload).hexdigest() != sums.get(name):
                    raise RuntimeError("R2 checkpoint checksum mismatch: " + name)
                destination.write_bytes(payload)
                r2_bundle_restored = True
                print("Restored verified R2 checkpoint:", name)
        print("R2 recovery bundle SHA-256 and ZIP CRC verified:", latest["sha256"])
    else:
        print("No prior native DREAM bundle found in R2; checking individual checkpoint backups.")

    # Recover a checkpoint uploaded after the last complete bundle (for example,
    # if Kaggle stopped before the packaging cell ran).
    for artifact_name, destination in (
        ("dream-native-pretrain-checkpoint.pt", PRETRAIN_CHECKPOINT),
        ("dream-native-sft-checkpoint.pt", SFT_CHECKPOINT)
    ):
        candidates = [x for x in listing.json().get("artifacts", [])
                      if x.get("name") == artifact_name
                      and x.get("created_at", "") > r2_bundle_created_at]
        if not candidates:
            continue
        latest_checkpoint = max(candidates, key=lambda x: x.get("created_at", ""))
        response = requests.get(
            DREAM_HOST + "/api/v1/notebook/artifacts/" + latest_checkpoint["sha256"],
            headers=R2_HEADERS, stream=True, timeout=(20, 1800)
        )
        response.raise_for_status()
        digest = hashlib.sha256()
        with destination.open("wb") as handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    digest.update(chunk)
                    handle.write(chunk)
        if digest.hexdigest() != latest_checkpoint["sha256"]:
            raise RuntimeError("R2 standalone checkpoint SHA-256 mismatch: " + artifact_name)
        print("Restored newer standalone R2 checkpoint:", artifact_name, latest_checkpoint["sha256"])
else:
    print("R2 restore unavailable: add DREAM_NOTEBOOK_ARTIFACT_TOKEN to Kaggle Secrets.")

# Reattach the previous private notebook output bundle to continue training
# across Kaggle sessions. Restore only known checkpoint paths and verify hashes.
prior_bundles = sorted(Path('/kaggle/input').glob('**/dream-training-artifacts.zip'))
if len(prior_bundles) > 1:
    raise RuntimeError('Attach at most one previous dream-training-artifacts.zip bundle.')
if prior_bundles and not r2_bundle_restored:
    with ZipFile(prior_bundles[0]) as archive:
        names = set(archive.namelist())
        if 'SHA256SUMS.txt' not in names:
            raise RuntimeError('Previous training bundle is missing SHA256SUMS.txt.')
        checksums = {}
        for line in archive.read('SHA256SUMS.txt').decode('utf-8').splitlines():
            digest, separator, archive_name = line.partition('  ')
            if separator and len(digest) == 64:
                checksums[archive_name] = digest
        for archive_name, destination in (
            ('checkpoints/dream-bpe.pt', PRETRAIN_CHECKPOINT),
            ('checkpoints/dream-bpe-instruct.pt', SFT_CHECKPOINT),
            ('checkpoints/dream-bpe-sft-base.pt', SFT_BASE_CHECKPOINT)
        ):
            if archive_name not in names:
                continue
            payload = archive.read(archive_name)
            if hashlib.sha256(payload).hexdigest() != checksums.get(archive_name):
                raise RuntimeError('Checkpoint checksum mismatch: ' + archive_name)
            destination.write_bytes(payload)
            print('Restored verified checkpoint:', archive_name)

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Stop here, open Notebook Settings, select a GPU accelerator, and run this cell again.')
print('GPU:', torch.cuda.get_device_name(0))


## 1. Attach the reviewed DREAM corpus

Create a **private Kaggle Dataset** containing a file named `approved-records.jsonl`, then attach it to this notebook with **Add Input**. Each line must contain `id`, `source`, `source_group`, `source_url` (HTTPS), `license`, `license_url` (HTTPS), `rights_basis`, `source_date` (`YYYY-MM-DD`), `rights_confirmed`, `privacy_reviewed`, `quality_reviewed`, `approval_granted`, `language`, and `text`.

The three human-review fields must be `true` only after real human review, and `approval_granted` must be `true` only after an explicit training-use decision. Use at least three independent source groups. Keep personal or confidential information out of the corpus. The audit below is a preflight, not approval.


In [ ]:
from pathlib import Path
import shutil

input_root = Path('/kaggle/input')
candidates = sorted(input_root.glob('**/approved-records.jsonl'))
if len(candidates) != 1:
    raise RuntimeError(
        'Attach exactly one private Kaggle Dataset containing approved-records.jsonl '
        'using Add Input. Found: ' + str(len(candidates))
    )
Path('data').mkdir(exist_ok=True)
shutil.copy2(candidates[0], 'data/approved-records.jsonl')
print('Selected input:', candidates[0].parent.name)
print('Uploaded bytes:', Path('data/approved-records.jsonl').stat().st_size)


In [ ]:
Path('eval-results/corpus-audit.json').unlink(missing_ok=True)
Path('eval-results/language-balance.json').unlink(missing_ok=True)
corpus_audit_run = subprocess.run([
    sys.executable, 'model/audit_corpus.py',
    '--input', 'data/approved-records.jsonl',
    '--out', 'eval-results/corpus-audit.json'
], capture_output=True, text=True)
corpus_audit = json.loads(Path('eval-results/corpus-audit.json').read_text())
print('Privacy and provenance preflight (not approval):')
print(json.dumps(corpus_audit, indent=2))
if corpus_audit['error_count']:
    raise RuntimeError('Corpus provenance audit has blocking errors; inspect the report and source records.')
if corpus_audit['possible_pii_findings']:
    raise RuntimeError('Potential personal data was detected; inspect and redact the source records before training.')
if corpus_audit['unreviewed_records']:
    raise RuntimeError('Every record must pass actual human rights, privacy and quality review before training.')
subprocess.run([sys.executable, 'model/prepare_corpus.py', '--input', 'data/approved-records.jsonl', '--output', 'data/prepared'], check=True)
import json
manifest = json.loads(Path('data/prepared/manifest.json').read_text())
print(json.dumps(manifest, indent=2))
subprocess.run([
    sys.executable, 'model/audit_language_balance.py',
    '--input', 'data/approved-records.jsonl',
    '--out', 'eval-results/language-balance.json',
    '--min-records', '1000', '--max-share', '0.5'
], check=True)
language_balance = json.loads(Path('eval-results/language-balance.json').read_text())
print('Language balance review (not an approval):')
print(json.dumps(language_balance, indent=2))
for split in ('train', 'validation', 'test'):
    size = Path(f'data/prepared/{split}.txt').stat().st_size
    print(split, 'UTF-8 bytes:', size)
    if size < 258:
        raise RuntimeError(f'{split} split is too small for the default 256-byte context. Add more reviewed data before training.')

## 2. Train DREAM from scratch

The starter settings use a small micro-batch plus gradient accumulation to reduce peak memory. Increase steps only if the available GPU and session can finish. Checkpoints are saved atomically under `/kaggle/working/checkpoints`. Re-running a cell in the same session resumes from its existing checkpoint; download the final artifact bundle before ending the session.


Each run uses a 240-second pretraining limit, then up to 60 seconds for SFT if an eligible reviewed SFT dataset exists. Both stages stop at optimizer-step boundaries and save resumable checkpoints.

In [ ]:
# Use DREAM's byte-fallback BPE tokenizer for multilingual training.
# Set to "byte" to reproduce legacy byte-tokenizer runs.
TOKENIZER = "bpe"
BPE_VOCAB_SIZE = 1024

TRAIN_STEPS = 2000
TRAIN_SECONDS = 240
MICRO_BATCH = 4
ACCUMULATION = 8
train_command = [
    sys.executable, 'model/train.py',
    '--data', 'data/prepared/train.txt',
    '--validation-data', 'data/prepared/validation.txt',
    '--corpus-manifest', 'data/prepared/manifest.json',
    '--out', str(PRETRAIN_CHECKPOINT),
    '--tokenizer', TOKENIZER,
    '--vocab-size', str(BPE_VOCAB_SIZE),
    '--steps', str(TRAIN_STEPS),
    '--max-seconds', str(TRAIN_SECONDS),
    '--batch-size', str(MICRO_BATCH),
    '--grad-accum-steps', str(ACCUMULATION),
    '--block-size', '256', '--d-model', '256', '--layers', '6', '--heads', '8',
    '--eval-every', '100', '--checkpoint-every', '250', '--eval-batches', '10'
]
if PRETRAIN_CHECKPOINT.exists():
    train_command += ['--resume', str(PRETRAIN_CHECKPOINT)]
    print('Resuming existing DREAM pretraining checkpoint with its saved tokenizer settings.')
MODEL_VOCAB_SIZE = BPE_VOCAB_SIZE if TOKENIZER == "bpe" else 256
subprocess.run([
    sys.executable, 'model/estimate_model.py',
    '--vocab-size', str(MODEL_VOCAB_SIZE),
    '--block-size', '256', '--d-model', '256',
    '--heads', '8', '--layers', '6',
    '--out', 'eval-results/model-budget.json'
], check=True)
budget = json.loads(Path('eval-results/model-budget.json').read_text())
print('Tokenizer:', TOKENIZER, 'model vocabulary size:', MODEL_VOCAB_SIZE)
print('Parameter count:', budget['parameter_count'])
print('FP32 Adam training-memory floor (bytes):',
      budget['parameter_memory']['fp32_adam_training_floor_bytes'])
print('Important:', budget['warning'])
subprocess.run(train_command, check=True)
# Persist the native pretraining checkpoint to the private DREAM R2 vault after each run.
r2_upload_file(PRETRAIN_CHECKPOINT, 'dream-native-pretrain-checkpoint.pt')

## 3. Evaluate on the held-out test split

This reports next-token cross-entropy per token and per represented UTF-8 byte and a sample generation. These metrics do not prove reasoning, factual accuracy, safety or instruction-following ability.

In [ ]:
subprocess.run([
    sys.executable, 'model/evaluate.py',
    '--checkpoint', str(PRETRAIN_CHECKPOINT),
    '--data', 'data/prepared/test.txt',
    '--batches', '50', '--batch-size', '8',
    '--out', 'eval-results/dream-bpe.json'
], check=True)
report = json.loads(Path('eval-results/dream-bpe.json').read_text())
print('Bits per byte:', report['bits_per_byte'])
print('Perplexity per byte:', report['perplexity_per_byte'])
print('Sample:', report['sample_generation'])

## 4. Optional: instruction-tune DREAM on reviewed conversations

This second stage starts from the DREAM checkpoint you just trained. It does not load another model. Provide separate `sft-train.jsonl` and `sft-validation.jsonl` files. Every record must include HTTPS `source_url` and `license_url`, a valid `rights_basis`, strict `source_date`, and human-completed rights/privacy/quality review plus explicit `approval_granted: true`. The script masks user/system prompts and learns only from assistant responses. Their source groups must be disjoint from each other and from the held-out pretraining test sources in `data/prepared/manifest.json`.

In [ ]:
from pathlib import Path
import shutil

input_root = Path('/kaggle/input')
train_candidates = sorted(input_root.glob('**/sft-train.jsonl'))
validation_candidates = sorted(input_root.glob('**/sft-validation.jsonl'))
sft_available = len(train_candidates) == 1 and len(validation_candidates) == 1
if sft_available:
    Path('data').mkdir(exist_ok=True)
    shutil.copy2(train_candidates[0], 'data/sft-train.jsonl')
    shutil.copy2(validation_candidates[0], 'data/sft-validation.jsonl')
    print('Training bytes:', Path('data/sft-train.jsonl').stat().st_size)
    print('Validation bytes:', Path('data/sft-validation.jsonl').stat().st_size)
else:
    print('Attach exactly one private Kaggle Dataset containing both sft-train.jsonl and sft-validation.jsonl to enable instruction tuning.')
    print('No complete SFT dataset found. Skipping instruction tuning; pretraining remains available.')


In [ ]:
if sft_available:
    for split_name in ('train', 'validation'):
        audit_path = f'eval-results/sft-{split_name}-corpus-audit.json'
        Path(audit_path).unlink(missing_ok=True)
        subprocess.run([
            sys.executable, 'model/audit_corpus.py',
            '--input', f'data/sft-{split_name}.jsonl',
            '--out', audit_path
        ], check=True, capture_output=True, text=True)
        sft_audit = json.loads(Path(audit_path).read_text())
        print(f'SFT {split_name} privacy/provenance audit (not approval):')
        print(json.dumps(sft_audit, indent=2))
        if (sft_audit['error_count'] or sft_audit['possible_pii_findings']
                or sft_audit['unreviewed_records']):
            raise RuntimeError(
                f'SFT {split_name} data requires provenance, privacy or human-review corrections.'
            )
    # The daily training budget is 300 seconds total: 240s pretraining + 60s SFT.
    import hashlib
    def sft_split_meta(split_name):
        split_path = Path(f'data/sft-{split_name}.jsonl')
        raw = split_path.read_bytes()
        return {
            'records': sum(1 for line in raw.splitlines() if line.strip()),
            'sha256': hashlib.sha256(raw).hexdigest()
        }
    sft_audit_manifest = {
        'schema': 'dream-sft-splits-v1',
        'dataset_sha256': hashlib.sha256(
            Path('data/sft-train.jsonl').read_bytes()
            + Path('data/sft-validation.jsonl').read_bytes()
        ).hexdigest(),
        'splits': {
            'train': sft_split_meta('train'),
            'validation': sft_split_meta('validation')
        }
    }
    sft_audit_manifest_path = Path('eval-results/sft-split-audit-manifest.json')
    sft_audit_manifest_path.write_text(
        json.dumps(sft_audit_manifest, indent=2), encoding='utf-8'
    )
    subprocess.run([
        sys.executable, 'data/verify_dream_ai_sft_audits.py',
        '--manifest', str(sft_audit_manifest_path),
        '--train-audit', 'eval-results/sft-train-corpus-audit.json',
        '--validation-audit', 'eval-results/sft-validation-corpus-audit.json',
        '--output', 'eval-results/sft-audit-gate.json'
    ], check=True, capture_output=True, text=True)
    print('SFT audit gate passed for the exact train/validation files.')

    SFT_STEPS = 1000
    SFT_SECONDS = 60
    # Pin the pretraining checkpoint used as the SFT base. Pretraining may
    # continue in later daily sessions, but SFT resume must use the exact same base hash.
    if not SFT_BASE_CHECKPOINT.exists():
        shutil.copy2(PRETRAIN_CHECKPOINT, SFT_BASE_CHECKPOINT)
        print('Pinned a verified DREAM pretraining snapshot for resumable SFT.')
    sft_command = [
        sys.executable, 'model/sft.py',
        '--checkpoint', str(SFT_BASE_CHECKPOINT),
        '--train-data', 'data/sft-train.jsonl',
        '--validation-data', 'data/sft-validation.jsonl',
        '--sft-audit-gate', 'eval-results/sft-audit-gate.json',
        '--pretraining-manifest', 'data/prepared/manifest.json',
        '--pretraining-test-data', 'data/prepared/test.txt',
        '--out', str(SFT_CHECKPOINT),
        '--steps', str(SFT_STEPS),
        '--max-seconds', str(SFT_SECONDS),
        '--batch-size', '2', '--grad-accum-steps', '4',
        '--eval-every', '100', '--checkpoint-every', '250',
        '--eval-batches', '10'
    ]
    if SFT_CHECKPOINT.exists():
        sft_command += ['--resume', str(SFT_CHECKPOINT)]
        print('Resuming existing DREAM instruction-tuning checkpoint.')
    subprocess.run(sft_command, check=True)
    # Persist the native instruction-tuning checkpoint independently of the final bundle.
    r2_upload_file(SFT_CHECKPOINT, 'dream-native-sft-checkpoint.pt')
    sft_completed = True
else:
    print('Instruction tuning skipped. The native pretraining checkpoint is still available.')

In [ ]:
if sft_completed:
    subprocess.run([
        sys.executable, 'model/evaluate.py',
        '--checkpoint', str(SFT_CHECKPOINT),
        '--data', 'data/prepared/test.txt',
        '--batches', '50', '--batch-size', '8',
        '--out', 'eval-results/dream-bpe-instruct.json'
    ], check=True)
    sft_report = json.loads(Path('eval-results/dream-bpe-instruct.json').read_text())
    print('Instruction checkpoint bits per byte:', sft_report['bits_per_byte'])
    print('Sample:', sft_report['sample_generation'])
else:
    print('No instruction checkpoint to evaluate; pretraining evaluation above is complete.')

## 6. Package the reproducible DREAM training artifacts

The bundle contains the checkpoints, evaluation reports, exact pretraining train/validation/test text splits and their manifest. If SFT completed, it also includes the reviewed SFT train/validation JSONL files and instruction checkpoint. The ZIP is written to `/kaggle/working/dream-training-artifacts.zip`; save a notebook version, then download it from the notebook's Output panel. Keep this bundle private if your reviewed data should not be shared.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from hashlib import sha256
artifact_entries = [
    (PRETRAIN_CHECKPOINT, 'checkpoints/dream-bpe.pt'),
    (Path('eval-results/dream-bpe.json'), 'eval-results/dream-bpe.json'),
    (Path('eval-results/language-balance.json'), 'eval-results/language-balance.json'),
    (Path('eval-results/corpus-audit.json'), 'eval-results/corpus-audit.json'),
    (Path('eval-results/model-budget.json'), 'eval-results/model-budget.json'),
    (Path('data/prepared/manifest.json'), 'data/prepared/manifest.json'),
    (Path('data/prepared/train.txt'), 'data/prepared/train.txt'),
    (Path('data/prepared/validation.txt'), 'data/prepared/validation.txt'),
    (Path('data/prepared/test.txt'), 'data/prepared/test.txt')
]
if sft_completed:
    artifact_entries += [
        (SFT_CHECKPOINT, 'checkpoints/dream-bpe-instruct.pt'),
        (SFT_BASE_CHECKPOINT, 'checkpoints/dream-bpe-sft-base.pt'),
        (Path('eval-results/dream-bpe-instruct.json'), 'eval-results/dream-bpe-instruct.json'),
        (Path('data/sft-train.jsonl'), 'data/sft-train.jsonl'),
        (Path('data/sft-validation.jsonl'), 'data/sft-validation.jsonl'),
        (Path('eval-results/sft-train-corpus-audit.json'), 'eval-results/sft-train-corpus-audit.json'),
        (Path('eval-results/sft-validation-corpus-audit.json'), 'eval-results/sft-validation-corpus-audit.json'),
        (Path('eval-results/sft-split-audit-manifest.json'), 'eval-results/sft-split-audit-manifest.json'),
        (Path('eval-results/sft-audit-gate.json'), 'eval-results/sft-audit-gate.json')
    ]
bundle = Path('/kaggle/working/dream-training-artifacts.zip')
with ZipFile(bundle, 'w', compression=ZIP_DEFLATED) as archive:
    for path, archive_name in artifact_entries:
        if not path.is_file():
            raise FileNotFoundError(f'Missing required training artifact: {path}')
        archive.write(path, arcname=archive_name)
    checksum_lines = []
    for path, archive_name in artifact_entries:
        digest = sha256(path.read_bytes()).hexdigest()
        checksum_lines.append(f'{digest}  {archive_name}')
    archive.writestr('SHA256SUMS.txt', '\n'.join(checksum_lines) + '\n')
print('Bundle size (bytes):', bundle.stat().st_size)
print('Artifact bundle:', bundle)
print('Download this file from the Kaggle notebook Output panel after saving a version.')

In [ ]:
# Optional: publish this private training bundle to the DREAM-owned artifact vault.
# This works only after DREAM HOST has a dedicated artifact token AND R2 storage configured.
from kaggle_secrets import UserSecretsClient
from pathlib import Path
import hashlib
import requests

artifact_bundle = Path("/kaggle/working/dream-training-artifacts.zip")
if not artifact_bundle.is_file():
    raise FileNotFoundError("Run the artifact packaging cell above first.")

try:
    artifact_token = UserSecretsClient().get_secret("DREAM_NOTEBOOK_ARTIFACT_TOKEN")
except Exception:
    artifact_token = ""

if not artifact_token:
    print("DREAM_NOTEBOOK_ARTIFACT_TOKEN is not configured in Kaggle Secrets.")
    print("Keep the ZIP private and download it from the Kaggle Output panel.")
else:
    digest = hashlib.sha256()
    with artifact_bundle.open("rb") as artifact_handle:
        for chunk in iter(lambda: artifact_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    artifact_sha256 = digest.hexdigest()
    host = "https://dream-host-live.onrender.com"
    with artifact_bundle.open("rb") as artifact_handle:
        response = requests.put(
            host + "/api/v1/notebook/artifacts/dream-training-artifacts.zip",
            data=artifact_handle,
            headers={
                "Authorization": "Bearer " + artifact_token,
                "Content-Type": "application/octet-stream"
            },
            timeout=(15, 1800)
        )
    response.raise_for_status()
    artifact_result = response.json()["artifact"]
    if artifact_result["sha256"] != artifact_sha256:
        raise RuntimeError("DREAM HOST artifact SHA-256 does not match the Kaggle bundle.")
    print("DREAM artifact stored and SHA-256 verified.")
    print("Bytes:", artifact_result["size_bytes"])
    print("SHA-256:", artifact_result["sha256"])
